# Notebook 05 — Digital Twin: Production Line Simulation

**Module:** `digital_twin/twin_simulator.py` and `digital_twin/twin_sync.py`  
**Model:** Discrete-event simulation of a 4-stage manufacturing line

---

## What is a Digital Twin?

A digital twin is a virtual replica of a physical system that:
1. **Mirrors** the real system state in real time
2. **Predicts** future states and failure modes
3. **Optimizes** control decisions before applying them to real hardware

This notebook demonstrates:
- Running the production line simulator
- Analyzing sensor streams and production statistics
- Computing OEE (Overall Equipment Effectiveness)
- Injecting faults and observing divergence detection

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from digital_twin.twin_simulator import TwinSimulator, StageConfig
from digital_twin.twin_sync import TwinSync
from evaluation.rl_metrics import compute_oee

print('Modules loaded.')

## 1. Configure and Run the Simulator

In [ ]:
# Define a 4-stage production line
stages = [
    StageConfig('machining',  nominal_cycle_time=30, cycle_time_std=2.0, failure_rate=0.005, mttr=60),
    StageConfig('assembly',   nominal_cycle_time=45, cycle_time_std=3.0, failure_rate=0.003, mttr=90),
    StageConfig('inspection', nominal_cycle_time=15, cycle_time_std=1.0, failure_rate=0.001, mttr=30),
    StageConfig('packaging',  nominal_cycle_time=20, cycle_time_std=1.5, failure_rate=0.002, mttr=45),
]

sim = TwinSimulator(
    stage_configs=stages,
    inter_stage_capacity=10,
    time_step=1.0,
    sensor_noise_std=0.02,
    random_seed=42,
)

print(f'Simulator: {sim.num_stages} stages, {sim.inter_stage_capacity} WIP buffer capacity')

In [ ]:
SIM_STEPS = 2000
history = sim.run(num_steps=SIM_STEPS, verbose=True)
df = pd.DataFrame(history)
print(f'Simulation complete. {len(df)} time steps.')
print(df.columns.tolist()[:10], '...')

## 2. Production Statistics

In [ ]:
summary = sim.summary()
for stage, stats in summary.items():
    if stage == 'total_sim_time_s':
        continue
    print(f"{'─'*50}")
    print(f"Stage: {stage}")
    for k, v in stats.items():
        print(f"  {k:25s} : {v}")

print(f"{'─'*50}")
print(f"Total simulation time: {summary.get('total_sim_time_s', 0):.0f}s")

## 3. Sensor Stream Visualization

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.3)
t = df['sim_time'].values

stage_colors = ['steelblue', 'darkorange', 'green', 'purple']

for i, (scfg, color) in enumerate(zip(stages, stage_colors)):
    prefix = f'stage_{i}_{scfg.name}'
    
    ax_temp = fig.add_subplot(gs[i, 0])
    ax_temp.plot(t, df[f'{prefix}_temperature'], linewidth=0.6, color=color, alpha=0.8)
    # Highlight fault periods
    fault_mask = df[f'{prefix}_status'] == 'repairing'
    ax_temp.fill_between(t, df[f'{prefix}_temperature'].min(), df[f'{prefix}_temperature'].max(),
                          where=fault_mask, alpha=0.2, color='red', label='Fault')
    ax_temp.set_ylabel(f'{scfg.name}\nTemperature (°C)', fontsize=9)
    ax_temp.grid(alpha=0.3)
    if i == 0:
        ax_temp.set_title('Temperature by Stage')
        ax_temp.legend(fontsize=8)
    
    ax_vib = fig.add_subplot(gs[i, 1])
    ax_vib.plot(t, df[f'{prefix}_vibration'], linewidth=0.6, color=color, alpha=0.8)
    ax_vib.fill_between(t, 0, df[f'{prefix}_vibration'].max(),
                         where=fault_mask, alpha=0.2, color='red')
    ax_vib.set_ylabel('Vibration', fontsize=9)
    ax_vib.grid(alpha=0.3)
    if i == 0:
        ax_vib.set_title('Vibration by Stage')

fig.text(0.5, 0.01, 'Simulation Time (s)', ha='center', fontsize=11)
plt.suptitle('Digital Twin — Sensor Streams (red = failure/repair)', fontsize=13)
plt.show()

## 4. Cumulative Production & Buffer Levels

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for i, (scfg, color) in enumerate(zip(stages, stage_colors)):
    prefix = f'stage_{i}_{scfg.name}'
    ax1.plot(t, df[f'{prefix}_total_produced'], label=scfg.name, color=color, linewidth=1.5)

ax1.set_ylabel('Cumulative Units Produced')
ax1.set_title('Production Output per Stage')
ax1.legend()
ax1.grid(alpha=0.3)

buffer_cols = [c for c in df.columns if c.startswith('buffer_')]
for bc in buffer_cols:
    idx = int(bc.split('_')[1])
    label = f'{stages[idx].name} → {stages[idx+1].name}'
    ax2.plot(t, df[bc], label=label, linewidth=1.2)

ax2.set_ylabel('WIP Buffer Level (units)')
ax2.set_xlabel('Simulation Time (s)')
ax2.set_title('Inter-Stage Buffer Levels')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. OEE Calculation

In [ ]:
print('=== OEE per Stage ===')
for i, scfg in enumerate(stages):
    prefix = f'stage_{i}_{scfg.name}'
    stage_stats = summary.get(prefix, {})
    
    availability = stage_stats.get('availability', 1.0)
    completed = stage_stats.get('total_produced', 0)
    
    oee = compute_oee(
        completed_jobs=completed,
        max_episode_steps=SIM_STEPS,
        num_machines=1,
        avg_processing_time=scfg.nominal_cycle_time,
        availability=availability,
    )
    print(f"  {scfg.name:15s} | OEE={oee['oee']:.3f} | "
          f"Avail={oee['availability']:.3f} | Perf={oee['performance']:.3f} | "
          f"Produced={completed}")

## 6. Export Simulation Data for Anomaly Detection

In [ ]:
output_path = Path('../logs/digital_twin/sim_output.csv')
saved = sim.export_csv(output_path)
print(f'Exported {len(df)} rows to {saved}')
print(f'Columns: {len(df.columns)}')
print(df.describe().round(2))

## 7. TwinSync in Replay Mode

TwinSync replays the exported CSV as if it were a live MQTT feed,
checking for divergence between the simulator and the 'real' readings.

In [ ]:
import time

sync = TwinSync(
    simulator=sim,
    protocol='replay',
    replay_csv=output_path,
    divergence_threshold=5.0,   # relaxed for replay (data comes from same sim)
)

sync.start()
print('TwinSync started in replay mode...')
time.sleep(2)  # let some messages arrive

status = sync.status()
print(f'Status: {status}')

sync.stop()
print('TwinSync stopped.')

## Summary

The digital twin provides:

| Capability | Description |
|-----------|-------------|
| Simulation | Stochastic discrete-event model of 4-stage production line |
| Sensor streams | Temperature, vibration, pressure with noise |
| Fault injection | Random breakdowns with repair times |
| OEE computation | Per-stage effectiveness analysis |
| Synchronization | MQTT/replay bridge to real hardware |
| Data export | CSV for training anomaly detection models |

**Next steps:**
- Connect real sensor feeds via MQTT (set `sync.protocol='mqtt'` and configure your broker)
- Train the RobotAnomalyDetector on exported sensor streams
- Use RL agent from Notebook 04 to optimize production scheduling inside the twin
- Run `python benchmarks/run_vision_benchmark.py` for production-level evaluation